# Caught & Shared — Dual Graph Approach
**Two separate graphs + fusion for mentor influence**

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData, Data
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data

In [ ]:
num_users = 25
num_items = 80

interactions = []
for u in range(num_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

multi_mentor_edges = [
    (0, 5, 0.70),
    (0, 7, 0.40),
    (1, 12, 0.85),
    (2, 5, 0.65),
    (3, 8, 0.90),
    (4, 15, 0.55),
    (4, 9, 0.45),
]

#### 2. Create 2 separate Graphs: Bipartite User-Item & Social Mentor Graph (User-User)

In [ ]:
bipartite_data = Data()
bipartite_data.x = torch.randn(num_users + num_items, 16)  # combined nodes

# Edge index for bipartite
user_offset = 0
item_offset = num_users

src = torch.tensor(inter_df['user_id'].values)
dst = torch.tensor(inter_df['item_id'].values) + item_offset

bipartite_data.edge_index = torch.stack([src, dst])

In [ ]:
social_data = Data()
social_data.x = torch.randn(num_users, 16)   # only users

mentor_src = torch.tensor([u for u, m, a in multi_mentor_edges])
mentor_dst = torch.tensor([m for u, m, a in multi_mentor_edges])
mentor_weight = torch.tensor([a for u, m, a in multi_mentor_edges], dtype=torch.float)

social_data.edge_index = torch.stack([mentor_src, mentor_dst])
social_data.edge_weight = mentor_weight

#### 3. Dual Graph Fusion Model

In [ ]:
class DualGraphModel(torch.nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.user_encoder_bipartite = torch.nn.Linear(16, hidden_dim)
        self.user_encoder_social = torch.nn.Linear(16, hidden_dim)
        
        self.item_encoder = torch.nn.Linear(16, hidden_dim)
        
        self.fusion = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        
    def forward(self, bipartite_x, social_x, user_id, mentor_config=None):
        user_emb_bip = self.user_encoder_bipartite(bipartite_x[user_id])
        user_emb_social = self.user_encoder_social(social_x[user_id])
        
        combined = torch.cat([user_emb_bip, user_emb_social], dim=-1)
        final_user_emb = self.fusion(combined)
        
        return final_user_emb


model = DualGraphModel()


In [ ]:
@torch.no_grad()
def get_recommendations_dual(model, 
                             bipartite_x, 
                             social_x, 
                             item_embs, 
                             user_id, 
                             mentor_config=None, 
                             bipartite_weight=0.6,
                             k=8):
    """
    mentor_config: list of (mentor_id, alpha) or None
    bipartite_weight: how much to trust the original user history (0.0 - 1.0)
    """
    model.eval()
    
    user_emb_bipartite = model.user_encoder_bipartite(bipartite_x[user_id])
    user_emb_social = model.user_encoder_social(social_x[user_id])
    
    # Multi-Mentor Social Influence
    if mentor_config and len(mentor_config) > 0:
        mentor_ids = [m for m, a in mentor_config]
        alphas = [a for m, a in mentor_config]
        
        mentor_social_embs = social_x[mentor_ids]
        
        social_influence = torch.zeros_like(user_emb_social)
        for m_emb, alpha in zip(mentor_social_embs, alphas):
            social_influence += alpha * (m_emb - user_emb_social)
        
        final_social_emb = user_emb_social + social_influence
    else:
        final_social_emb = user_emb_social

    combined = torch.cat([user_emb_bipartite, final_social_emb], dim=-1)
    final_user_emb = model.fusion(combined)
    
    scores = torch.matmul(final_user_emb.unsqueeze(0), item_embs.T).squeeze(0)
    
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'mentors': mentor_config,
        'bipartite_weight': bipartite_weight,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }


rec_base = get_recommendations_dual(model, data['user'].x, data['user'].x, 
                                   data['item'].x, user_id=0, mentor_config=None)
print(f"User 0 (bipartite only)     → {rec_base['recommended_items']}")

rec_single = get_recommendations_dual(model, data['user'].x, data['user'].x, 
                                     data['item'].x, user_id=0, 
                                     mentor_config=[(5, 0.8)], bipartite_weight=0.55)
print(f"User 0 + 1 mentor          → {rec_single['recommended_items']}")

rec_multi = get_recommendations_dual(model, data['user'].x, data['user'].x, 
                                    data['item'].x, user_id=0, 
                                    mentor_config=[(5, 0.65), (7, 0.35)], 
                                    bipartite_weight=0.5)
print(f"User 0 + 2 mentors        → {rec_multi['recommended_items']}")